# 06 Insurance Claim Relationship Graph Construction
## Graph-Enhanced Insurance Claim Fraud Detection

**Project Scope:** Academic & Research Pipeline  
**Dataset Provenance:** Relational tables (`data/relational/*.csv`)  
**Reproducibility:** 100% deterministic graph construction, zero random numbers, evidence-based integrity validation  

---

### Objectives
1. **Entity Extraction (Nodes):** Parse Claims, Claimants, Policies, Vehicles, Providers, Invoices, and Locations.
2. **Relationship Modeling (Edges):** Construct semantic edges (`OWNS`, `FILED`, `COVERED_BY`, `ASSOCIATED_WITH`, `INVOLVES`, `HAS`, `OCCURRED_AT`, `LOCATED_AT`, `ISSUED_BY`, `WITHIN_TERRITORY`).
3. **Graph Integrity & Validation:** Rigorously test referential integrity, absence of orphan edges, absence of accidental self-loops, and endpoint type consistency.
4. **Topological Statistics:** Compute entity counts, relationship distributions, degree distributions, and connected components.
5. **Artifact Generation:** Persist `data/graph/nodes.csv`, `data/graph/edges.csv`, and validation/statistics reports in `reports/`.


In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import networkx as nx

from src.graph.build_graph import build_insurance_graph, extract_nodes, extract_edges
from src.graph.validate_graph import validate_graph, validate_graph_files
from src.graph.statistics import compute_graph_statistics, generate_graph_statistics_report

print('All required modules loaded successfully.')


All required modules loaded successfully.


### 1. Ingestion of Relational Data Source Tables
We inspect the underlying relational tables from `data/relational/` to confirm entity and foreign key availability.


In [2]:
relational_dir = Path('data/relational')
tables = {f.stem: pd.read_csv(f) for f in sorted(relational_dir.glob('*.csv'))}
for name, df in tables.items():
    print(f'Table: {name:<12} | Rows: {len(df):<5} | Columns: {len(df.columns)}')


Table: claimants    | Rows: 120   | Columns: 6
Table: claims       | Rows: 320   | Columns: 12
Table: invoices     | Rows: 220   | Columns: 5
Table: locations    | Rows: 60    | Columns: 4
Table: policies     | Rows: 140   | Columns: 7
Table: providers    | Rows: 25    | Columns: 5
Table: vehicles     | Rows: 130   | Columns: 6


### 2. Graph Construction: Nodes and Edges
We extract all nodes (with attributes serialized to structured JSON) and all semantic relationships.


In [3]:
nodes_df, edges_df, G = build_insurance_graph(
    relational_dir='data/relational',
    output_dir='data/graph'
)
print(f'Constructed Graph:')
print(f'  Total Nodes: {len(nodes_df)}')
print(f'  Total Edges: {len(edges_df)}')
print(f'  NetworkX Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}')


Constructed Graph:
  Total Nodes: 1020
  Total Edges: 2615
  NetworkX Nodes: 1020, Edges: 2615


### 3. Inspection of Graph Node and Edge Tables
Let us inspect the schema and samples from `nodes.csv` and `edges.csv`.


In [4]:
print('--- Nodes Head ---')
print(nodes_df[['node_id', 'node_type', 'source_id']].head(7))
print('\nSample Node Attribute JSON:')
print(json.loads(nodes_df.iloc[0]['attributes']))

print('\n--- Edges Head ---')
print(edges_df.head(7))


--- Nodes Head ---
          node_id node_type source_id
0  claim:CLM00001     Claim  CLM00001
1  claim:CLM00002     Claim  CLM00002
2  claim:CLM00003     Claim  CLM00003
3  claim:CLM00004     Claim  CLM00004
4  claim:CLM00005     Claim  CLM00005
5  claim:CLM00006     Claim  CLM00006
6  claim:CLM00007     Claim  CLM00007

Sample Node Attribute JSON:
{'claim_amount': 132760, 'claim_date': '2026-01-26', 'claim_type': 'Theft', 'description': 'Claim description 1', 'fraud_label': 1, 'status': 'Open'}

--- Edges Head ---
            source              target      relationship  weight
0  location:loc001      location:surat  WITHIN_TERRITORY     1.0
1  location:loc002  location:ahmedabad  WITHIN_TERRITORY     1.0
2  location:loc003      location:delhi  WITHIN_TERRITORY     1.0
3  location:loc004  location:ahmedabad  WITHIN_TERRITORY     1.0
4  location:loc005       location:pune  WITHIN_TERRITORY     1.0
5  location:loc006      location:surat  WITHIN_TERRITORY     1.0
6  location:loc007  loc

### 4. Graph Structural & Semantic Validation
We verify referential integrity, absence of orphan edges, absence of unintended self-loops, and endpoint type consistency.


In [5]:
val_report = validate_graph_files(
    nodes_path='data/graph/nodes.csv',
    edges_path='data/graph/edges.csv',
    report_path='reports/graph_validation.json'
)
print(f'Overall Validation Status: {"PASSED" if val_report["is_valid"] else "FAILED"}')
print(f'Errors Count: {len(val_report["errors"])}')
print('\nValidation Check Details:')
for name, res in val_report['checks'].items():
    status = 'PASS' if res['passed'] else 'FAIL'
    print(f'  [{status}] {name}')


Overall Validation Status: PASSED
Errors Count: 0

Validation Check Details:
  [PASS] required_columns
  [PASS] node_id_uniqueness
  [PASS] referential_integrity
  [PASS] no_self_loops
  [PASS] relationship_consistency
  [PASS] attribute_json_validity


### 5. Graph Topological Statistics & Distributions
We compute entity breakdowns, relationship frequencies, degree distributions, and connected components.


In [6]:
stats = generate_graph_statistics_report(
    nodes_path='data/graph/nodes.csv',
    edges_path='data/graph/edges.csv',
    output_path='reports/graph_statistics.json'
)

print('--- Nodes by Type ---')
for k, v in stats['nodes_by_type'].items():
    print(f'  {k:<12}: {v:>5}')

print('\n--- Edges by Relationship ---')
for k, v in stats['edges_by_relationship'].items():
    print(f'  {k:<18}: {v:>5}')

print('\n--- Degree Distribution ---')
deg_stats = stats['degree_distribution']
print(f'  Min: {deg_stats["min"]}, Max: {deg_stats["max"]}, Mean: {deg_stats["mean"]}, Median: {deg_stats["median"]}, Std: {deg_stats["std"]}')
print(f'  25th Percentile: {deg_stats["p25"]}, 75th Percentile: {deg_stats["p75"]}')

print('\n--- Degree by Entity Type ---')
for ntype, info in deg_stats['degree_by_node_type'].items():
    print(f'  {ntype:<12}: Mean Degree = {info["mean_degree"]:>5.2f}, Max = {info["max_degree"]}')

print('\n--- Top 5 High-Degree Hub Nodes ---')
for hub in deg_stats['top_degree_nodes'][:5]:
    print(f'  {hub["node_id"]} ({hub["node_type"]}) -> Degree {hub["degree"]}')

print('\n--- Connected Components ---')
cc = stats['connected_components']
print(f'  Number of Components: {cc["number_of_components"]}')
print(f'  Largest Component Size: {cc["largest_component_size"]} ({cc["largest_component_ratio"] * 100:.1f}% of total graph)')


--- Nodes by Type ---
  Claim       :   320
  Invoice     :   220
  Policy      :   140
  Vehicle     :   130
  Claimant    :   120
  Location    :    65
  Provider    :    25

--- Edges by Relationship ---
  FILED             :   320
  COVERED_BY        :   320
  HAS               :   320
  OCCURRED_AT       :   320
  INVOLVES          :   320
  ASSOCIATED_WITH   :   320
  OWNS              :   270
  ISSUED_BY         :   220
  LOCATED_AT        :   145
  WITHIN_TERRITORY  :    60

--- Degree Distribution ---
  Min: 1, Max: 123, Mean: 5.13, Median: 4.0, Std: 7.93
  25th Percentile: 2.0, 75th Percentile: 6.0

--- Degree by Entity Type ---
  Claim       : Mean Degree =  6.00, Max = 6
  Invoice     : Mean Degree =  2.45, Max = 7
  Policy      : Mean Degree =  3.29, Max = 8
  Vehicle     : Mean Degree =  3.46, Max = 9
  Claimant    : Mean Degree =  5.92, Max = 11
  Location    : Mean Degree =  9.00, Max = 123
  Provider    : Mean Degree = 22.60, Max = 34

--- Top 5 High-Degree Hub Nodes -

### 6. Subgraph Inspection: High-Volume Provider Ecosystem
Let us extract and inspect a 1-hop ego network around one of the top providers to examine real relationship interconnectivity.


In [7]:
top_provider = deg_stats['top_degree_nodes'][0]['node_id']
ego_nodes = list(G.neighbors(top_provider)) + [top_provider]
ego_subgraph = G.subgraph(ego_nodes)
print(f'Ego Subgraph around {top_provider}:')
print(f'  Total Subgraph Nodes: {ego_subgraph.number_of_nodes()}')
print(f'  Total Subgraph Edges: {ego_subgraph.number_of_edges()}')

ego_node_types = pd.Series([G.nodes[n].get('node_type', 'Unknown') for n in ego_nodes]).value_counts()
print('\nEgo Neighborhood Composition:')
print(ego_node_types.to_dict())


Ego Subgraph around location:mumbai:
  Total Subgraph Nodes: 124
  Total Subgraph Edges: 212

Ego Neighborhood Composition:
{'Claim': 77, 'Claimant': 26, 'Location': 17, 'Provider': 4}


### 7. Key Findings & Phase 6 Conclusion

1. **Graph Completeness:**
   - Exactly **1,020 nodes** across 7 distinct entity types (`Claim`, `Invoice`, `Policy`, `Vehicle`, `Claimant`, `Location`, `Provider`).
   - Exactly **2,615 edges** capturing 10 distinct semantic relationships.
2. **Strict Referential Integrity:**
   - Zero orphan edge endpoints (all sources and targets exist in `nodes.csv`).
   - Zero accidental self-loops (`source != target` universally).
   - 100% relationship endpoint consistency matching the insurance domain schema.
3. **Connected Topology:**
   - The network constitutes a single connected component (100% coverage, 0 isolated nodes), uniting territory locations, providers, and claimants.
4. **Reproducibility:**
   - The entire graph construction pipeline is deterministic and reproducible via `src/graph/build_graph.py`.
   - Validation and statistics reports are persisted to `reports/graph_validation.json` and `reports/graph_statistics.json`.
